<a href="https://colab.research.google.com/github/tasmeer-Siddiqui125/flyrank-ml-internship/blob/main/w04_baseline_score_cleaned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmeer-Siddiqui125/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds one transparent baseline for the Search Intelligence classification lane. The rule uses only signals available at the **2026-03-15 decision moment**.

The future outcome window is used only to audit the signals. It is **not** used as an input to the baseline score.

The baseline is decision-support, not causal proof. It creates a simple benchmark that a later Week-5 model can be compared against.


## Setup

Week 3 defined the honest frame:

- Decision-time features: **2026-03-01 through 2026-03-15**
- Future outcome window: **2026-03-16 through 2026-03-31**
- Label: `future_engagement`

The baseline score must not use `future_engagement`, `future_engagement_rate`, future-period performance, or any label-derived field.


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print("DuckDB connected to Hugging Face")


## 1. Signal checks before the rule

Two signals are checked before encoding the rule:

1. **`gsc_impressions`** — linked to the FlyRank **quick-win** idea because search visibility creates an observable opportunity to review content.
2. **`gsc_avg_position`** — linked to the FlyRank **CTR-fix/search-position** logic because position gives context for interpreting search performance.

The bucket tables show `n` and the observed future-engagement positive rate.

The verdicts are descriptive:

- `CONFIRMED` = consistent movement in the expected direction
- `OPPOSITE` = consistent movement in the opposite direction
- `MIXED` = not consistently monotonic
- `FALSE` = no usable signal / insufficient evidence

A `MIXED` result is acceptable; it means the signal should be used cautiously rather than described as confirmed.


In [ ]:
# Build the honest March decision frame.
# Features: March 1-15. Label: March 16-31.
# The future label is created only for signal auditing.

honest_frame = con.execute(f"""
WITH features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(
            CASE WHEN gsc_data_available IS TRUE
            THEN gsc_impressions ELSE 0 END
        ) AS gsc_impressions,
        SUM(
            CASE WHEN gsc_data_available IS TRUE
            THEN gsc_clicks ELSE 0 END
        ) AS gsc_clicks,
        AVG(
            CASE
                WHEN gsc_data_available IS TRUE
                     AND gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS gsc_avg_position,
        SUM(
            CASE WHEN ga4_data_available IS TRUE
            THEN ga4_sessions ELSE 0 END
        ) AS ga4_sessions,
        SUM(
            CASE WHEN ga4_data_available IS TRUE
            THEN ga4_engaged_sessions ELSE 0 END
        ) AS ga4_engaged_sessions
    FROM read_parquet('{path}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),
future_content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(
            CASE WHEN ga4_data_available IS TRUE
            THEN ga4_sessions ELSE 0 END
        ) AS future_sessions,
        SUM(
            CASE WHEN ga4_data_available IS TRUE
            THEN ga4_engaged_sessions ELSE 0 END
        ) AS future_engaged_sessions
    FROM read_parquet('{path}')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
),
labeled AS (
    SELECT
        client_hash_id,
        content_hash_id,
        future_engaged_sessions * 1.0 / future_sessions
            AS future_engagement_rate
    FROM future_content
    WHERE future_sessions > 0
),
threshold AS (
    SELECT MEDIAN(future_engagement_rate) AS median_rate
    FROM labeled
    WHERE future_engagement_rate > 0
)
SELECT
    f.*,
    l.future_engagement_rate,
    CASE
        WHEN l.future_engagement_rate > 0
             AND l.future_engagement_rate >= t.median_rate
        THEN 1
        ELSE 0
    END AS future_engagement
FROM features f
INNER JOIN labeled l
    ON f.client_hash_id = l.client_hash_id
   AND f.content_hash_id = l.content_hash_id
CROSS JOIN threshold t
""").df()

print("Honest frame shape:", honest_frame.shape)
print("Label distribution:")
print(honest_frame["future_engagement"].value_counts().sort_index())


In [ ]:
# Signal 1: gsc_impressions

signal1 = honest_frame.copy()

signal1["impression_bucket"] = pd.cut(
    signal1["gsc_impressions"],
    bins=[-1, 100, 500, 2000, np.inf],
    labels=["0-100", "101-500", "501-2000", "2000+"]
)

impression_table = (
    signal1.groupby("impression_bucket", observed=False)
    .agg(
        n=("future_engagement", "size"),
        positive_rate=("future_engagement", "mean")
    )
    .reset_index()
)

print("Signal 1: gsc_impressions")
display(impression_table)

rates = impression_table["positive_rate"].dropna().tolist()

if len(rates) < 2:
    impression_verdict = "FALSE"
elif all(rates[i] <= rates[i + 1] for i in range(len(rates) - 1)) and rates[-1] > rates[0]:
    impression_verdict = "CONFIRMED"
elif all(rates[i] >= rates[i + 1] for i in range(len(rates) - 1)) and rates[-1] < rates[0]:
    impression_verdict = "OPPOSITE"
else:
    impression_verdict = "MIXED"

print("Verdict:", impression_verdict)


In [ ]:
# Signal 2: gsc_avg_position
# Position 0 is treated as missing/no valid position, not rank zero.

signal2 = honest_frame[honest_frame["gsc_avg_position"].notna()].copy()

signal2["position_bucket"] = pd.cut(
    signal2["gsc_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

position_table = (
    signal2.groupby("position_bucket", observed=False)
    .agg(
        n=("future_engagement", "size"),
        positive_rate=("future_engagement", "mean")
    )
    .reset_index()
)

print("Signal 2: gsc_avg_position")
display(position_table)

rates = position_table["positive_rate"].dropna().tolist()

if len(rates) < 2:
    position_verdict = "FALSE"
elif all(rates[i] >= rates[i + 1] for i in range(len(rates) - 1)) and rates[0] > rates[-1]:
    position_verdict = "CONFIRMED"
elif all(rates[i] <= rates[i + 1] for i in range(len(rates) - 1)) and rates[0] < rates[-1]:
    position_verdict = "OPPOSITE"
else:
    position_verdict = "MIXED"

print("Verdict:", position_verdict)


### Signal interpretation

The observed March results are used as an audit, not as causal evidence.

Both signals are allowed to be `MIXED`. The rule below therefore treats them as **transparent prioritization inputs**, not as proven predictors.

The `gsc_impressions` threshold of 500 and the position range of 11–50 are deliberately simple baseline thresholds. They are not learned from the future label.


## 2. Encode ONE rule and build the ranked queue

### One baseline rule

Prioritize content that has meaningful search visibility and is in a reviewable search-position range.

**Score**

- `+2` if `gsc_impressions >= 500`
- `+1` if `gsc_avg_position` is between **11 and 50**
- Otherwise, add `0`

Maximum score = **3**.

### One reason code

`visible_needs_position_review`

### One action label

`review_search_performance`

Rows with score `0` receive:

- reason code: `not_prioritized`
- action: `monitor`

The score uses only decision-time fields. No future outcome or label-derived field enters the calculation.


In [ ]:
# Build the ranked queue from decision-time fields only.
# No future outcome or label-derived field is used in this calculation.

decision_columns = [
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

decision_frame = honest_frame[decision_columns].copy()

queue = decision_frame.copy()

queue["score"] = (
    (queue["gsc_impressions"] >= 500).astype(int) * 2
    + queue["gsc_avg_position"].between(
        11, 50, inclusive="both"
    ).fillna(False).astype(int)
)

queue["reason_code"] = np.where(
    queue["score"] > 0,
    "visible_needs_position_review",
    "not_prioritized"
)

queue["action"] = np.where(
    queue["score"] > 0,
    "review_search_performance",
    "monitor"
)

queue = queue.sort_values(
    ["score", "gsc_impressions", "content_hash_id"],
    ascending=[False, False, True]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "score",
    "reason_code",
    "action"
]

os.makedirs("work/outputs", exist_ok=True)

queue[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows ranked:", len(queue))
print("CSV written: work/outputs/baseline_action_score.csv")
display(queue.head(10)[output_cols])


## 3. Top-10 review

Each top-ten row is reviewed with:

- **action** — what the baseline recommends
- **why it is here** — which rule conditions produced the score
- **what would make it wrong** — a concrete limitation or missing context

This is a skeptical review of the rule, not a claim that the recommendation is correct.


In [ ]:
top10 = queue.head(10).copy()

review_rows = []

for _, row in top10.iterrows():
    visibility_met = row["gsc_impressions"] >= 500
    position_met = (
        pd.notna(row["gsc_avg_position"])
        and 11 <= row["gsc_avg_position"] <= 50
    )

    reasons = []

    if visibility_met:
        reasons.append(
            f"it has {row['gsc_impressions']:.0f} impressions"
        )

    if position_met:
        reasons.append(
            f"its average position is {row['gsc_avg_position']:.1f}"
        )

    if reasons:
        why = " and ".join(reasons) + " under the baseline rule."
    else:
        why = "it received a non-zero baseline score."

    if visibility_met and position_met:
        wrong = (
            "The visibility may come from low-value queries, and the "
            "11–50 position may not represent a stable or actionable opportunity."
        )
    elif visibility_met:
        wrong = (
            "The content has visibility, but position is missing or outside "
            "the review range, so the rule may be over-prioritizing volume."
        )
    else:
        wrong = (
            "The rule has limited context about query quality, business value, "
            "content quality, and position stability."
        )

    review_rows.append({
        "rank": int(row["rank"]),
        "content_hash_id": row["content_hash_id"],
        "action": row["action"],
        "why_it_is_here": why,
        "what_would_make_it_wrong": wrong
    })

review = pd.DataFrame(review_rows)

display(review)


## 4. Weak picks + leakage check

Weak picks are items that receive a non-zero score but may still be poor recommendations because the baseline cannot see query quality, content quality, business value, or stability of the observed position.

The baseline score uses only:

- `gsc_impressions`
- `gsc_avg_position`

The future outcome is retained only in `honest_frame` for the earlier signal audit.


In [ ]:
print("Leakage check")
print("- Decision-time inputs:", ["gsc_impressions", "gsc_avg_position"])
print("- Future outcome used in score:", False)
print("- Label-derived input used in score:", False)
print("- Future-period performance used in score:", False)
print("- Client/content IDs used as predictive measurements:", False)

weak_picks = (
    queue[queue["score"] > 0]
    .sort_values(["score", "gsc_impressions"], ascending=[True, True])
    .head(5)
)

print("\nExample weak/low-priority scored picks:")
display(weak_picks[output_cols])


## Self-check

- [x] Two signal bucket tables are visible and include `n`.
- [x] Signal 1 has a one-word verdict.
- [x] Signal 2 has a one-word verdict.
- [x] At least one signal is linked to a real FlyRank flag idea.
- [x] One rule is used for the baseline.
- [x] The rule has one score, one reason code, and one action label.
- [x] `work/outputs/baseline_action_score.csv` is generated by the notebook.
- [x] Ten rows are reviewed with action, why they are there, and what would make them wrong.
- [x] No future-window or label-derived field is used by the score.
- [ ] Run the complete notebook top-to-bottom before committing the executed version.
- [ ] Keep the generated CSV out of git; commit the notebook and permitted receipts only.

**Observed in the uploaded notebook:** both `gsc_impressions` and `gsc_avg_position` produced `MIXED` verdicts. The cleaned notebook recalculates these results when executed.
